<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="../zh-CN/Deep-Learning/16-gans-diffusion-flow-matching.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to Deep Learning guideline](Deep-Learning.html)

## **GANs, Diffusion Models, and Flow Matching** {#gans-diffusion-flow-matching}

GANs, diffusion models, and flow matching all transform a simple source of randomness into structured data, but they learn different objects. A GAN learns a generator through an adversarial critic. A diffusion model learns to reverse progressive corruption. Flow matching learns a time-dependent velocity field whose ODE transports a source distribution into the data distribution.

![GANs, diffusion models, and flow matching use different training signals and sampling paths.](assets/dl16-family-map.svg){fig-align="center" width="78%" fig-alt="Three panels compare adversarial one-pass generation, iterative diffusion denoising, and ODE flow matching."}

*Original synthesis based on [Generative Adversarial Nets](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html), [DDPM](https://arxiv.org/abs/2006.11239), and [Flow Matching](https://arxiv.org/abs/2210.02747).*

The executable thread uses scikit-learn's copy of the [UCI Optical Recognition of Handwritten Digits dataset](https://archive.ics.uci.edu/dataset/80/optical%2Brecognition%2Bof%2B), DOI [10.24432/C50P49](https://doi.org/10.24432/C50P49), CC BY 4.0. A fixed 70/15/15 split is shared by every model. A classifier trained only on real training images supplies one domain-specific feature space for conditional fidelity, confidence, coverage, and nearest-training diagnostics. It is a probe, not an impartial universal evaluator.

<details>
<summary><strong>PyTorch: establish the shared data, split, and evaluation probe</strong></summary>

```python
import copy
import math
import random

import numpy as np
import torch
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

torch.set_num_threads(1)


def seed_everything(seed=1616):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


seed_everything()
digits = load_digits()
all_images = torch.tensor(digits.images, dtype=torch.float32).flatten(1) / 16.0
all_labels = torch.tensor(digits.target, dtype=torch.long)
indices = np.arange(len(all_images))
train_idx, holdout_idx = train_test_split(
    indices, test_size=0.30, random_state=1616, stratify=digits.target
)
val_idx, test_idx = train_test_split(
    holdout_idx, test_size=0.50, random_state=1616,
    stratify=digits.target[holdout_idx],
)
train_x, train_y = all_images[train_idx], all_labels[train_idx]
val_x, val_y = all_images[val_idx], all_labels[val_idx]
test_x, test_y = all_images[test_idx], all_labels[test_idx]
train_scaled = train_x * 2 - 1
val_scaled = val_x * 2 - 1
test_scaled = test_x * 2 - 1


class DigitProbe(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.head = nn.Linear(32, 10)

    def forward(self, images, return_features=False):
        features = self.features(images)
        logits = self.head(features)
        return (logits, features) if return_features else logits


seed_everything(1617)
probe = DigitProbe()
optimizer = torch.optim.AdamW(probe.parameters(), lr=3e-3, weight_decay=1e-4)
for _ in range(65):
    optimizer.zero_grad()
    loss = F.cross_entropy(probe(train_x), train_y)
    loss.backward()
    optimizer.step()
probe.eval()
with torch.no_grad():
    probe_accuracy = float((probe(test_x).argmax(1) == test_y).float().mean())


def audit_samples(images_01, intended_labels=None):
    images_01 = images_01.detach().clamp(0, 1)
    with torch.no_grad():
        logits, features = probe(images_01, return_features=True)
        probabilities = logits.softmax(1)
        nearest = torch.cdist(images_01, train_x).min(1).values
        real_features = probe(test_x, return_features=True)[1]
    rounded_unique = torch.unique(torch.round(images_01 * 8) / 8, dim=0).shape[0] / len(images_01)
    result = {
        "confidence": float(probabilities.max(1).values.mean()),
        "rounded unique ratio": rounded_unique,
        "nearest-train distance": float(nearest.mean()),
        "feature mean gap": float((features.mean(0) - real_features.mean(0)).norm()),
    }
    if intended_labels is not None:
        result["conditional accuracy"] = float((probabilities.argmax(1) == intended_labels).float().mean())
        result["covered predicted classes"] = int(probabilities.argmax(1).unique().numel())
    return result


assert all_images.shape == (1797, 64)
assert len(set(train_idx) & set(test_idx)) == 0
assert probe_accuracy > 0.90
print({"split": (len(train_x), len(val_x), len(test_x)), "probe accuracy": round(probe_accuracy, 3),
       "pixel range": (float(train_scaled.min()), float(train_scaled.max()))})
```

</details>

The images are only 8x8, so generated samples demonstrate objectives and failure modes rather than photorealistic synthesis. All quality claims remain local to this dataset, architecture, seed, and short CPU training budget.


### **Adversarial Generation** {#adversarial-generation}

A GAN contains a generator $G_{\theta}(z,c)$ and discriminator $D_{\phi}(x,c)$. The generator maps noise $z\sim p(z)$, optionally with condition $c$, into a sample. The discriminator learns evidence that distinguishes real from generated pairs. Neither model sees an explicit pointwise likelihood; the critic converts a distribution discrepancy into gradients for the generator.

![A generator and discriminator receive opposing losses in a coupled game.](assets/dl16-gan-game.svg){fig-align="center" width="76%" fig-alt="Noise and a condition enter a generator; real and generated observations enter a discriminator; opposing losses update the two networks."}

*Original mechanism diagram based on [Goodfellow et al.](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html).*

Training is not ordinary minimization of one stationary loss. As $G$ changes, the discriminator's task changes; as $D$ changes, the generator's gradient field changes. Update ratio, optimizer, normalization, capacity, and data augmentation can therefore alter the game even when the written objective is unchanged.

<details>
<summary><strong>PyTorch: define and train one conditional adversarial generator</strong></summary>

```python
class ConditionalGenerator(nn.Module):
    def __init__(self, noise_dim=20, embedding_dim=10):
        super().__init__()
        self.noise_dim = noise_dim
        self.label_embedding = nn.Embedding(10, embedding_dim)
        self.network = nn.Sequential(
            nn.Linear(noise_dim + embedding_dim, 96), nn.LeakyReLU(0.2),
            nn.Linear(96, 128), nn.LeakyReLU(0.2), nn.Linear(128, 64), nn.Tanh(),
        )

    def forward(self, noise, labels):
        return self.network(torch.cat([noise, self.label_embedding(labels)], dim=1))


class ConditionalDiscriminator(nn.Module):
    def __init__(self, embedding_dim=10):
        super().__init__()
        self.label_embedding = nn.Embedding(10, embedding_dim)
        self.network = nn.Sequential(
            nn.Linear(64 + embedding_dim, 128), nn.LeakyReLU(0.2),
            nn.Linear(128, 64), nn.LeakyReLU(0.2), nn.Linear(64, 1),
        )

    def forward(self, images, labels):
        return self.network(torch.cat([images, self.label_embedding(labels)], dim=1)).squeeze(1)


seed_everything(1620)
gan_generator = ConditionalGenerator()
gan_discriminator = ConditionalDiscriminator()
g_optimizer = torch.optim.Adam(gan_generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
d_optimizer = torch.optim.Adam(gan_discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))
generator = torch.Generator().manual_seed(1620)
gan_trace = []
for step in range(850):
    batch_indices = torch.randint(len(train_scaled), (128,), generator=generator)
    real, labels = train_scaled[batch_indices], train_y[batch_indices]
    noise = torch.randn((128, gan_generator.noise_dim), generator=generator)
    fake = gan_generator(noise, labels)

    d_optimizer.zero_grad()
    real_loss = F.binary_cross_entropy_with_logits(
        gan_discriminator(real, labels), torch.full((128,), 0.9)
    )
    fake_loss = F.binary_cross_entropy_with_logits(
        gan_discriminator(fake.detach(), labels), torch.zeros(128)
    )
    discriminator_loss = real_loss + fake_loss
    discriminator_loss.backward()
    d_optimizer.step()

    g_optimizer.zero_grad()
    generator_loss = F.binary_cross_entropy_with_logits(
        gan_discriminator(fake, labels), torch.ones(128)
    )
    generator_loss.backward()
    g_optimizer.step()
    if step % 100 == 0:
        gan_trace.append((step, float(discriminator_loss.detach()), float(generator_loss.detach())))

evaluation_labels = torch.arange(10).repeat_interleave(20)
with torch.no_grad():
    gan_noise = torch.randn((len(evaluation_labels), gan_generator.noise_dim),
                            generator=torch.Generator().manual_seed(1621))
    gan_samples_scaled = gan_generator(gan_noise, evaluation_labels)
gan_samples = (gan_samples_scaled + 1) / 2
assert gan_samples.shape == (200, 64)
print({"final discriminator loss": round(float(discriminator_loss.detach()), 3),
       "final generator loss": round(float(generator_loss.detach()), 3),
       "recorded checkpoints": len(gan_trace)})
```

</details>

The generated tensor range and finite losses are only implementation checks. A discriminator loss near $\log 4$ can mean equilibrium, underfitting, or matched confusion; it is not a quality score. Samples and coverage diagnostics are required.


### **The GAN Minimax Objective** {#gan-minimax-objective}

The original game is

$$
\min_G\max_D\;V(D,G)=
\mathbb{E}_{x\sim p_{\text{data}}}\log D(x)
+\mathbb{E}_{z\sim p(z)}\log(1-D(G(z))).
$$

For fixed $G$, the optimal discriminator is

$$
D^{*}(x)=\frac{p_{\text{data}}(x)}{p_{\text{data}}(x)+p_g(x)}.
$$

Substitution yields a Jensen-Shannon divergence objective up to a constant. This theoretical result assumes an optimal discriminator and unrestricted function classes; alternating finite neural updates do not satisfy those assumptions.

In early training, a strong discriminator makes $D(G(z))\approx0$. The minimax generator loss $\log(1-D(G(z)))$ then saturates. The non-saturating alternative $-\log D(G(z))$ has the same desired fixed point but stronger gradients.

![Saturating and non-saturating generator losses have different early gradients.](assets/dl16-minimax-gradients.svg){fig-align="center" width="76%" fig-alt="Two loss curves compare vanishing gradients of the saturating objective with stronger gradients of the non-saturating objective."}

*Original gradient comparison derived from the [GAN paper](https://papers.nips.cc/paper_files/paper/2014/hash/f033ed80deb0234979a61f95710dbe25-Abstract.html).*

<details>
<summary><strong>PyTorch: compare generator gradient magnitudes on the trained discriminator</strong></summary>

```python
gan_discriminator.eval()
demo_noise = torch.randn((128, gan_generator.noise_dim), generator=torch.Generator().manual_seed(1630))
demo_labels = train_y[:128]
demo_fake = gan_generator(demo_noise, demo_labels)
demo_logits = gan_discriminator(demo_fake, demo_labels)

saturating_loss = -F.binary_cross_entropy_with_logits(demo_logits, torch.zeros_like(demo_logits))
non_saturating_loss = F.binary_cross_entropy_with_logits(demo_logits, torch.ones_like(demo_logits))
saturating_gradient = torch.autograd.grad(saturating_loss, demo_fake, retain_graph=True)[0].norm(dim=1).mean()
non_saturating_gradient = torch.autograd.grad(non_saturating_loss, demo_fake)[0].norm(dim=1).mean()

assert saturating_gradient > 0 and non_saturating_gradient > 0
print({"mean D(fake)": round(float(demo_logits.sigmoid().mean().detach()), 3),
       "saturating input-gradient norm": round(float(saturating_gradient.detach()), 5),
       "non-saturating input-gradient norm": round(float(non_saturating_gradient.detach()), 5)})
```

</details>

Gradient magnitude depends on current logits and parameterization. The non-saturating loss improves signal but does not solve cycling, mode collapse, overfitting, or poor conditioning by itself.


### **Conditional GANs and Architectural Improvements** {#conditional-gans-architectural-improvements}

A conditional GAN gives $c$ to both players:

$$
G(z,c)\rightarrow x,\qquad D(x,c)\rightarrow \mathbb{R}.
$$

The discriminator must reject both unrealistic images and mismatched image-condition pairs. [Conditional GANs](https://arxiv.org/abs/1411.1784) originally demonstrated class-controlled digit generation. Modern implementations may use projection discriminators, conditional normalization, attention, residual blocks, spectral normalization, and carefully matched convolutional up/downsampling.

![Both generator and discriminator receive the condition, and label intervention tests whether it is used.](assets/dl16-conditional-gan.svg){fig-align="center" width="76%" fig-alt="Noise and label enter a generator; generated image and label enter a discriminator; an audit changes only the label."}

*Original conditional-contract diagram based on [Mirza and Osindero](https://arxiv.org/abs/1411.1784).*

<details>
<summary><strong>PyTorch: intervene on labels while holding generator noise fixed</strong></summary>

```python
gan_generator.eval()
fixed_noise = torch.randn((10, gan_generator.noise_dim), generator=torch.Generator().manual_seed(1640))
label_order_a = torch.arange(10)
label_order_b = torch.roll(label_order_a, shifts=1)
with torch.no_grad():
    samples_a = (gan_generator(fixed_noise, label_order_a) + 1) / 2
    samples_b = (gan_generator(fixed_noise, label_order_b) + 1) / 2
    intervention_distance = (samples_a - samples_b).pow(2).mean(dim=1).sqrt()
    predicted_a = probe(samples_a).argmax(1)
    predicted_b = probe(samples_b).argmax(1)

assert float(intervention_distance.mean()) > 0
print({"mean image change after label intervention": round(float(intervention_distance.mean()), 3),
       "predictions changed": int((predicted_a != predicted_b).sum()),
       "out of": len(predicted_a)})
```

</details>

An output changing with $c$ proves sensitivity, not correct control. Conditional accuracy, within-condition diversity, and condition leakage must all be measured. In text-to-image systems, this extends to prompt adherence, compositionality, and unintended correlations.


### **Wasserstein GANs** {#wasserstein-gans}

When data and generator distributions lie on thin, disjoint manifolds, divergences used by the original GAN can yield poor gradients. WGAN replaces the probability discriminator with a real-valued 1-Lipschitz critic and optimizes the Kantorovich-Rubinstein dual:

$$
W_1(p_r,p_g)=\sup_{\lVert f\rVert_L\le1}
\mathbb{E}_{p_r}[f(x)]-\mathbb{E}_{p_g}[f(x)].
$$

[WGAN](https://arxiv.org/abs/1701.07875) originally enforced the constraint with weight clipping. [WGAN-GP](https://papers.nips.cc/paper_files/paper/2017/hash/892c3b1c6dccd52936e27cbd0ff683d6-Abstract.html) penalizes critic gradient norms on points interpolated between real and generated samples:

$$
\lambda\,\mathbb{E}_{\hat{x}}(\lVert\nabla_{\hat{x}}D(\hat{x})\rVert_2-1)^2.
$$

![A 1-Lipschitz critic provides a distance-shaped signal, with gradient penalty imposed on interpolated points.](assets/dl16-wgan.svg){fig-align="center" width="76%" fig-alt="Real and generated distributions connect through a one-Lipschitz critic whose input-gradient norm is penalized."}

*Original structural diagram based on [WGAN](https://arxiv.org/abs/1701.07875) and [WGAN-GP](https://papers.nips.cc/paper_files/paper/2017/hash/892c3b1c6dccd52936e27cbd0ff683d6-Abstract.html).*

<details>
<summary><strong>PyTorch: train a conditional WGAN-GP on the same split</strong></summary>

```python
class ConditionalCritic(ConditionalDiscriminator):
    pass


def gradient_penalty(critic, real, fake, labels, generator):
    interpolation = torch.rand((len(real), 1), generator=generator)
    mixed = (interpolation * real + (1 - interpolation) * fake).requires_grad_(True)
    score = critic(mixed, labels)
    gradient = torch.autograd.grad(score.sum(), mixed, create_graph=True)[0]
    return (gradient.norm(2, dim=1) - 1).pow(2).mean()


seed_everything(1650)
wgan_generator = ConditionalGenerator()
wgan_critic = ConditionalCritic()
wg_optimizer = torch.optim.Adam(wgan_generator.parameters(), lr=1e-4, betas=(0.0, 0.9))
wc_optimizer = torch.optim.Adam(wgan_critic.parameters(), lr=1e-4, betas=(0.0, 0.9))
generator = torch.Generator().manual_seed(1650)
for generator_step in range(360):
    for _ in range(3):
        batch_indices = torch.randint(len(train_scaled), (128,), generator=generator)
        real, labels = train_scaled[batch_indices], train_y[batch_indices]
        noise = torch.randn((128, wgan_generator.noise_dim), generator=generator)
        fake = wgan_generator(noise, labels).detach()
        wc_optimizer.zero_grad()
        penalty = gradient_penalty(wgan_critic, real, fake, labels, generator)
        critic_loss = wgan_critic(fake, labels).mean() - wgan_critic(real, labels).mean() + 10 * penalty
        critic_loss.backward()
        wc_optimizer.step()
    labels = train_y[torch.randint(len(train_y), (128,), generator=generator)]
    noise = torch.randn((128, wgan_generator.noise_dim), generator=generator)
    wg_optimizer.zero_grad()
    wgan_loss = -wgan_critic(wgan_generator(noise, labels), labels).mean()
    wgan_loss.backward()
    wg_optimizer.step()

with torch.no_grad():
    wgan_noise = torch.randn((len(evaluation_labels), wgan_generator.noise_dim),
                             generator=torch.Generator().manual_seed(1651))
    wgan_samples = (wgan_generator(wgan_noise, evaluation_labels) + 1) / 2

assert wgan_samples.shape == gan_samples.shape
print({"critic loss": round(float(critic_loss.detach()), 3),
       "gradient penalty": round(float(penalty.detach()), 3),
       "generator loss": round(float(wgan_loss.detach()), 3)})
```

</details>

The trained critic objective is not automatically an accurate numerical estimate of $W_1$: finite capacity, imperfect optimization, and approximate Lipschitz enforcement matter. Its practical value is the generator gradient and training diagnostic, not a guaranteed transport certificate.


### **Mode Collapse and Training Instability** {#mode-collapse-training-instability}

Mode collapse maps many $z$ values to a small subset of valid outputs. Those samples can have high fidelity while the generator misses other data modes. Oscillation occurs because each player chases a moving opponent; discriminator overfitting can expose useless directions; vanishing or exploding gradients can halt learning.

![Mode collapse can preserve local realism while losing distribution coverage.](assets/dl16-mode-collapse.svg){fig-align="center" width="76%" fig-alt="Several data modes are compared with a generator covering one mode; diagnostics separate fidelity, coverage, and memorization."}

*Original failure-mode diagram.*

Common mitigations include balanced capacity and update ratios, non-saturating or Wasserstein objectives, gradient penalties, spectral normalization, minibatch features, data augmentation, and multiple generators. None removes the need to inspect samples across conditions and seeds.

<details>
<summary><strong>PyTorch: audit GAN fidelity, coverage, diversity, and copying</strong></summary>

```python
gan_audit = audit_samples(gan_samples, evaluation_labels)
wgan_audit = audit_samples(wgan_samples, evaluation_labels)
for name, row in {"non-saturating GAN": gan_audit, "WGAN-GP": wgan_audit}.items():
    print({name: {key: round(value, 3) if isinstance(value, float) else value
                  for key, value in row.items()}})

assert 0 <= gan_audit["conditional accuracy"] <= 1
assert 1 <= wgan_audit["covered predicted classes"] <= 10
```

</details>

This probe can be confidently wrong on generated artifacts. Rounded uniqueness can miss semantic collapse, while nearest-pixel distance can miss memorization after small transformations. Reliable studies add repeated seeds, class-conditional precision/recall, feature-space coverage, train-data extraction tests, and human review.


### **The Diffusion Forward Process** {#diffusion-forward-process}

DDPM defines a fixed Markov corruption process:

$$
q(x_t\mid x_{t-1})=\mathcal{N}(\sqrt{1-\beta_t}\,x_{t-1},\beta_t I).
$$

With $\alpha_t=1-\beta_t$ and $\bar{\alpha}_t=\prod_{s=1}^{t}\alpha_s$, any timestep can be sampled directly:

$$
x_t=\sqrt{\bar{\alpha}_t}x_0+\sqrt{1-\bar{\alpha}_t}\epsilon,
\qquad \epsilon\sim\mathcal{N}(0,I).
$$

![Forward diffusion progressively reduces signal-to-noise ratio and admits direct sampling at any timestep.](assets/dl16-forward-diffusion.svg){fig-align="center" width="78%" fig-alt="A clean sample becomes increasingly noisy, with the closed-form equation shown beside the chain."}

*Original process diagram based on [DDPM](https://arxiv.org/abs/2006.11239).*

The schedule controls signal-to-noise ratio (SNR). Too little terminal noise leaves a train-sample mismatch; too aggressive early noise destroys useful learning signal. Modern schedules are often described through log SNR rather than raw $\beta_t$.

<details>
<summary><strong>PyTorch: construct and verify the closed-form forward process</strong></summary>

```python
diffusion_steps = 40
betas = torch.linspace(1e-4, 0.14, diffusion_steps)
alphas = 1 - betas
alpha_bars = torch.cumprod(alphas, dim=0)


def extract(values, timesteps, target):
    return values[timesteps].view(-1, *([1] * (target.ndim - 1)))


def q_sample(clean, timesteps, noise):
    signal = extract(alpha_bars.sqrt(), timesteps, clean)
    noise_scale = extract((1 - alpha_bars).sqrt(), timesteps, clean)
    return signal * clean + noise_scale * noise


generator = torch.Generator().manual_seed(1660)
forward_clean = test_scaled[:180]
forward_noise = torch.randn(forward_clean.shape, generator=generator)
forward_statistics = {}
for timestep in (0, 9, 19, 39):
    t = torch.full((len(forward_clean),), timestep, dtype=torch.long)
    noisy = q_sample(forward_clean, t, forward_noise)
    forward_statistics[timestep] = {
        "alpha_bar": float(alpha_bars[timestep]),
        "correlation": float(torch.corrcoef(torch.stack([forward_clean.flatten(), noisy.flatten()]))[0, 1]),
        "std": float(noisy.std()),
    }
print({t: {k: round(v, 3) for k, v in row.items()} for t, row in forward_statistics.items()})
assert forward_statistics[0]["correlation"] > forward_statistics[39]["correlation"]
```

</details>

The forward process is not learned in standard DDPM. Its tractable posterior structure creates supervised denoising targets at randomly selected noise levels.


### **Learning the Reverse Denoising Process** {#learning-reverse-denoising-process}

The exact reverse conditional $q(x_{t-1}\mid x_t)$ depends on the unknown data distribution. A neural model approximates its mean or an equivalent noise/score target while a chosen variance schedule controls stochasticity. Time embeddings tell the network which SNR regime it is solving; condition embeddings supply class, text, or another control.

For classifier-free guidance later, the same network is trained conditionally and unconditionally. A fraction of labels are replaced by a learned null condition. This is deliberate condition dropout, not missing-data leakage.

<details>
<summary><strong>PyTorch: train one conditional multi-noise denoiser with condition dropout</strong></summary>

```python
def time_embedding(timesteps, dimension=24):
    half = dimension // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half) / max(half - 1, 1))
    angles = timesteps.float().unsqueeze(1) * frequencies.unsqueeze(0)
    return torch.cat([angles.sin(), angles.cos()], dim=1)


class DiffusionDenoiser(nn.Module):
    def __init__(self, data_dim=64, hidden=128, label_dim=20):
        super().__init__()
        self.data_dim = data_dim
        self.null_label = 10
        self.label_embedding = nn.Embedding(11, label_dim)
        self.network = nn.Sequential(
            nn.Linear(data_dim + 24 + label_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, data_dim),
        )

    def forward(self, noisy, timesteps, labels):
        context = torch.cat([noisy, time_embedding(timesteps), self.label_embedding(labels)], dim=1)
        return self.network(context)


seed_everything(1670)
diffusion_model = DiffusionDenoiser()
optimizer = torch.optim.AdamW(diffusion_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1670)
diffusion_loss_trace = []
for step in range(1500):
    batch_indices = torch.randint(len(train_scaled), (192,), generator=generator)
    clean, labels = train_scaled[batch_indices], train_y[batch_indices].clone()
    timesteps = torch.randint(diffusion_steps, (len(clean),), generator=generator)
    noise = torch.randn(clean.shape, generator=generator)
    noisy = q_sample(clean, timesteps, noise)
    drop_condition = torch.rand(len(clean), generator=generator) < 0.15
    labels[drop_condition] = diffusion_model.null_label
    optimizer.zero_grad()
    predicted_noise = diffusion_model(noisy, timesteps, labels)
    diffusion_loss = F.mse_loss(predicted_noise, noise)
    diffusion_loss.backward()
    torch.nn.utils.clip_grad_norm_(diffusion_model.parameters(), 1.0)
    optimizer.step()
    if step % 150 == 0:
        diffusion_loss_trace.append(float(diffusion_loss.detach()))

diffusion_model.eval()
with torch.no_grad():
    val_t = torch.randint(diffusion_steps, (len(val_scaled),), generator=torch.Generator().manual_seed(1671))
    val_noise = torch.randn(val_scaled.shape, generator=torch.Generator().manual_seed(1672))
    val_noisy = q_sample(val_scaled, val_t, val_noise)
    val_noise_mse = float(F.mse_loss(diffusion_model(val_noisy, val_t, val_y), val_noise))

assert val_noise_mse < 1.0
print({"validation noise MSE": round(val_noise_mse, 3),
       "first recorded loss": round(diffusion_loss_trace[0], 3),
       "last recorded loss": round(diffusion_loss_trace[-1], 3)})
```

</details>

Uniform timestep sampling weights all indices equally, not all SNR regimes equally. Loss weighting, architecture, self-conditioning, learned variance, and data parameterization can shift where model capacity is spent.


### **DDPM Objectives and Parameterizations** {#ddpm-objectives-parameterizations}

The simplified DDPM objective predicts the sampled noise:

$$
\mathcal{L}_{\epsilon}=\mathbb{E}_{x_0,t,\epsilon}
\lVert\epsilon-\epsilon_{\theta}(x_t,t,c)\rVert_2^2.
$$

Equivalent targets include clean data $x_0$ and velocity

$$
v_t=\sqrt{\bar{\alpha}_t}\epsilon-
\sqrt{1-\bar{\alpha}_t}x_0.
$$

![Noise, clean-data, and velocity prediction are algebraically related under a known schedule.](assets/dl16-reverse-diffusion.svg){fig-align="center" width="78%" fig-alt="A denoiser receives noisy data, timestep, and condition, then predicts epsilon, x0, or velocity."}

*Original parameterization diagram based on [DDPM](https://arxiv.org/abs/2006.11239).*

Although parameterizations are algebraically convertible, their optimization scales differ across SNR. Clipping an $x_0$ estimate, weighting by SNR, and predicting variance change the practical model. “Same objective” should mean same weighting and schedule, not only a convertible target.

<details>
<summary><strong>PyTorch: verify epsilon, x0, and v conversions at arbitrary timesteps</strong></summary>

```python
generator = torch.Generator().manual_seed(1680)
parameter_clean = test_scaled[:96]
parameter_t = torch.randint(diffusion_steps, (len(parameter_clean),), generator=generator)
parameter_noise = torch.randn(parameter_clean.shape, generator=generator)
parameter_noisy = q_sample(parameter_clean, parameter_t, parameter_noise)
a = extract(alpha_bars.sqrt(), parameter_t, parameter_clean)
b = extract((1 - alpha_bars).sqrt(), parameter_t, parameter_clean)
velocity = a * parameter_noise - b * parameter_clean
clean_from_velocity = a * parameter_noisy - b * velocity
noise_from_velocity = b * parameter_noisy + a * velocity
clean_from_noise = (parameter_noisy - b * parameter_noise) / a

assert torch.allclose(clean_from_velocity, parameter_clean, atol=2e-5)
assert torch.allclose(noise_from_velocity, parameter_noise, atol=2e-5)
assert torch.allclose(clean_from_noise, parameter_clean, atol=2e-5)
print({"max x0 reconstruction error": float((clean_from_velocity - parameter_clean).abs().max()),
       "max epsilon reconstruction error": float((noise_from_velocity - parameter_noise).abs().max())})
```

</details>

These identities are important when converting checkpoints or schedulers. A mismatch in prediction type can produce plausible early denoising followed by severe divergence.


### **Score-Based Models, SDEs, and Probability-Flow ODEs** {#score-sde-probability-flow-ode}

For variance-preserving diffusion, an epsilon predictor gives a score estimate

$$
s_{\theta}(x_t,t)\approx\nabla_{x_t}\log p_t(x_t)
=-\frac{\epsilon_{\theta}(x_t,t)}{\sqrt{1-\bar{\alpha}_t}}.
$$

In continuous time, a forward SDE $dx=f(x,t)dt+g(t)dW_t$ has reverse-time SDE

$$
dx=[f(x,t)-g(t)^2\nabla_x\log p_t(x)]dt+g(t)d\bar{W}_t,
$$

integrated from noise toward data. The probability-flow ODE removes the Brownian term and uses half the score correction. With an exact score, it has the same marginal distributions as the SDE, but individual paths and numerical errors differ.

![The reverse SDE is stochastic while the probability-flow ODE is deterministic for fixed initial noise.](assets/dl16-sde-ode.svg){fig-align="center" width="78%" fig-alt="Stochastic reverse-SDE trajectories and deterministic probability-flow ODE trajectories share time marginals under an exact score."}

*Original comparison based on [Score-Based Generative Modeling through SDEs](https://openreview.net/pdf?id=PxTIG12RRHS).*

<details>
<summary><strong>PyTorch: convert noise prediction to score and separate drift from stochasticity</strong></summary>

```python
diffusion_model.eval()
generator = torch.Generator().manual_seed(1690)
sde_clean = test_scaled[:128]
sde_t = torch.full((len(sde_clean),), 24, dtype=torch.long)
sde_noise = torch.randn(sde_clean.shape, generator=generator)
sde_noisy = q_sample(sde_clean, sde_t, sde_noise)
with torch.no_grad():
    predicted_epsilon = diffusion_model(sde_noisy, sde_t, test_y[:128])
noise_scale = extract((1 - alpha_bars).sqrt(), sde_t, sde_noisy)
estimated_score = -predicted_epsilon / noise_scale

beta = betas[24]
forward_drift = -0.5 * beta * sde_noisy
reverse_sde_drift = forward_drift - beta * estimated_score
probability_flow_drift = forward_drift - 0.5 * beta * estimated_score
stochastic_increment = math.sqrt(float(beta)) * torch.randn(
    sde_noisy.shape, generator=torch.Generator().manual_seed(1691)
)

assert torch.allclose(reverse_sde_drift - forward_drift,
                      2 * (probability_flow_drift - forward_drift), atol=1e-6)
print({"mean score norm": round(float(estimated_score.norm(dim=1).mean()), 3),
       "reverse drift norm": round(float(reverse_sde_drift.norm(dim=1).mean()), 3),
       "stochastic increment std": round(float(stochastic_increment.std()), 3)})
```

</details>

The code checks local algebra, not a continuous-time solver. SDE/ODE likelihood and sampling require a consistent continuous schedule, numerical integrator, and error control. Reusing a discrete DDPM network without matching conventions can invalidate the interpretation.


### **Classifier and Classifier-Free Guidance** {#classifier-classifier-free-guidance}

Classifier guidance adds $\nabla_{x_t}\log p(c\mid x_t)$ from a noise-aware classifier to the unconditional score. It requires a separate classifier trained across noise levels and can exploit classifier errors.

Classifier-free guidance (CFG) trains one denoiser with and without conditions, then combines predictions:

$$
\epsilon_{\mathrm{cfg}}=epsilon_{u}
+w(\epsilon_{c}-\epsilon_{u}).
$$

$w=0$ is unconditional; $w=1$ is the ordinary conditional prediction; $w>1$ extrapolates toward the condition. Higher guidance often improves adherence and apparent fidelity while reducing diversity or causing oversaturation.

![Classifier-free guidance combines unconditional and conditional predictions with a tunable extrapolation scale.](assets/dl16-guidance.svg){fig-align="center" width="78%" fig-alt="Unconditional and conditional epsilon predictions combine into a guided prediction controlled by scale w."}

*Original CFG diagram based on [Ho and Salimans](https://openreview.net/pdf/ea628d03c92a49b54bc2d757d209e024e7885980.pdf).*

<details>
<summary><strong>PyTorch: implement deterministic DDIM sampling with CFG</strong></summary>

```python
@torch.no_grad()
def guided_epsilon(model, noisy, timestep, labels, guidance_scale):
    t = torch.full((len(noisy),), timestep, dtype=torch.long)
    conditional = model(noisy, t, labels)
    null_labels = torch.full_like(labels, model.null_label)
    unconditional = model(noisy, t, null_labels)
    return unconditional + guidance_scale * (conditional - unconditional)


@torch.no_grad()
def ddim_sample(model, labels, step_count=20, guidance_scale=1.0, seed=1700, data_dim=64):
    generator = torch.Generator().manual_seed(seed)
    sample = torch.randn((len(labels), data_dim), generator=generator)
    schedule = torch.linspace(diffusion_steps - 1, 0, step_count).round().long().unique_consecutive()
    for index, timestep_tensor in enumerate(schedule):
        timestep = int(timestep_tensor)
        epsilon = guided_epsilon(model, sample, timestep, labels, guidance_scale)
        alpha_bar = alpha_bars[timestep]
        predicted_clean = ((sample - torch.sqrt(1 - alpha_bar) * epsilon) /
                           torch.sqrt(alpha_bar)).clamp(-1, 1)
        next_timestep = int(schedule[index + 1]) if index + 1 < len(schedule) else -1
        next_alpha_bar = alpha_bars[next_timestep] if next_timestep >= 0 else torch.tensor(1.0)
        sample = torch.sqrt(next_alpha_bar) * predicted_clean + torch.sqrt(1 - next_alpha_bar) * epsilon
    return sample


guidance_samples = {}
for offset, scale in enumerate((0.0, 1.0, 3.0)):
    generated = ddim_sample(diffusion_model, evaluation_labels, step_count=20,
                            guidance_scale=scale, seed=1700)
    guidance_samples[scale] = (generated + 1) / 2
    row = audit_samples(guidance_samples[scale], evaluation_labels)
    print({"guidance": scale, **{key: round(value, 3) if isinstance(value, float) else value
                                 for key, value in row.items()}})

assert all(samples.shape == (200, 64) for samples in guidance_samples.values())
```

</details>

CFG is part of the deployed sampler, so its scale and schedule must be versioned. The best value depends on condition type, model, negative prompt or null embedding, and evaluation objective.


### **Latent Diffusion** {#latent-diffusion}

Pixel-space diffusion repeatedly evaluates a large network at full spatial resolution. Latent diffusion first learns an encoder $E$ and decoder $D$, then models $z=E(x)$:

$$
x\xrightarrow{E}z,qquad z_T\rightarrow\cdots\rightarrow z_0,qquad \hat{x}=D(z_0).
$$

![Latent diffusion moves repeated denoising into a compressed representation between an encoder and decoder.](assets/dl16-latent-diffusion.svg){fig-align="center" width="78%" fig-alt="An image is encoded, iteratively denoised in a smaller latent space, and decoded back to pixels."}

*Original architecture diagram based on [Latent Diffusion Models](https://arxiv.org/abs/2112.10752).*

Compression reduces compute and can emphasize perceptual structure, but the generator cannot recover information discarded by the autoencoder. Reconstruction quality, latent scaling, decoder artifacts, and diffusion quality are separate failure sources.

<details>
<summary><strong>PyTorch: train a compact autoencoder and a diffusion model in its latent space</strong></summary>

```python
class LatentAutoencoder(nn.Module):
    def __init__(self, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(64, 48), nn.SiLU(), nn.Linear(48, latent_dim))
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 48), nn.SiLU(), nn.Linear(48, 64), nn.Tanh())

    def forward(self, images):
        latent = self.encoder(images)
        return self.decoder(latent), latent


seed_everything(1710)
latent_autoencoder = LatentAutoencoder()
optimizer = torch.optim.AdamW(latent_autoencoder.parameters(), lr=2e-3, weight_decay=1e-5)
for _ in range(75):
    optimizer.zero_grad()
    reconstruction, _ = latent_autoencoder(train_scaled)
    reconstruction_loss = F.mse_loss(reconstruction, train_scaled)
    reconstruction_loss.backward()
    optimizer.step()
latent_autoencoder.eval()
with torch.no_grad():
    train_latent_raw = latent_autoencoder.encoder(train_scaled)
    test_reconstruction = latent_autoencoder(test_scaled)[0]
latent_mean = train_latent_raw.mean(0)
latent_std = train_latent_raw.std(0).clamp_min(1e-5)
train_latent = (train_latent_raw - latent_mean) / latent_std

latent_diffusion_model = DiffusionDenoiser(data_dim=16, hidden=96)
optimizer = torch.optim.AdamW(latent_diffusion_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1710)
for _ in range(900):
    batch_indices = torch.randint(len(train_latent), (192,), generator=generator)
    clean, labels = train_latent[batch_indices], train_y[batch_indices].clone()
    timesteps = torch.randint(diffusion_steps, (len(clean),), generator=generator)
    noise = torch.randn(clean.shape, generator=generator)
    noisy = q_sample(clean, timesteps, noise)
    labels[torch.rand(len(labels), generator=generator) < 0.15] = latent_diffusion_model.null_label
    optimizer.zero_grad()
    loss = F.mse_loss(latent_diffusion_model(noisy, timesteps, labels), noise)
    loss.backward()
    optimizer.step()

latent_generated = ddim_sample(latent_diffusion_model, evaluation_labels, step_count=20,
                               guidance_scale=1.5, seed=1711, data_dim=16)
with torch.no_grad():
    latent_unscaled = latent_generated * latent_std + latent_mean
    latent_samples = (latent_autoencoder.decoder(latent_unscaled) + 1) / 2
latent_audit = audit_samples(latent_samples, evaluation_labels)
print({"autoencoder test MSE": round(float(F.mse_loss(test_reconstruction, test_scaled)), 4),
       "pixel dimension": 64, "latent dimension": 16,
       **{key: round(value, 3) if isinstance(value, float) else value for key, value in latent_audit.items()}})
assert latent_samples.shape == (200, 64)
```

</details>

This compact autoencoder is trained only for the chapter. Production latent diffusion commonly uses perceptual and adversarial reconstruction objectives, spatial latent tensors, and carefully calibrated scaling. Compute savings should include encoder/decoder cost and memory, not only latent dimension.


### **Diffusion Sampling and Acceleration** {#diffusion-sampling-acceleration}

Ancestral DDPM sampling adds posterior noise at each reverse step. DDIM constructs a non-Markovian process with the same training objective and permits deterministic trajectories. Subsampling timesteps reduces neural function evaluations (NFEs), but coarse integration accumulates model and discretization error.

Acceleration families include higher-order ODE/SDE solvers, timestep optimization, progressive distillation, consistency models, latent-space generation, and caching. Wall-clock speed also depends on network architecture, batch size, memory movement, and hardware; NFE is not a complete latency metric.

<details>
<summary><strong>PyTorch: compare full and reduced DDIM schedules under one model</strong></summary>

```python
accelerated_results = {}
for offset, steps in enumerate((40, 20, 8)):
    samples_scaled = ddim_sample(diffusion_model, evaluation_labels, step_count=steps,
                                 guidance_scale=1.5, seed=1720)
    samples = (samples_scaled + 1) / 2
    accelerated_results[steps] = audit_samples(samples, evaluation_labels)
    print({"requested NFE": steps,
           **{key: round(value, 3) if isinstance(value, float) else value
              for key, value in accelerated_results[steps].items()}})

assert set(accelerated_results) == {40, 20, 8}
```

</details>

Common initial noise makes this a paired comparison of timestep schedules. Fewer steps can still score better under a short-trained model because repeated biased denoising updates accumulate error; that is not evidence that coarse sampling is universally superior. A rigorous sampler benchmark adds repeated seeds, measured latency, confidence intervals, and task-specific quality metrics.


### **Flow Matching and Rectified Flow** {#flow-matching-rectified-flow}

Continuous normalizing flows define

$$
\frac{dz_t}{dt}=v_{\theta}(z_t,t,c).
$$

Flow matching avoids solving this ODE during training. Choose a conditional probability path between source $x_0$ and data $x_1$. For the straight interpolation used here,

$$
x_t=(1-t)x_0+tx_1,\qquad u_t=x_1-x_0,
$$

and train $v_{\theta}(x_t,t,c)$ to regress $u_t$. Generation samples $x_0$ and numerically integrates the learned field to $t=1$.

![Flow matching samples endpoints, regresses path velocity without training-time simulation, then integrates an ODE for generation.](assets/dl16-flow-matching.svg){fig-align="center" width="78%" fig-alt="Noise and data endpoints define a linear path and target velocity; generation integrates a learned ODE."}

*Original process diagram based on [Flow Matching](https://arxiv.org/abs/2210.02747) and [Rectified Flow](https://openreview.net/pdf?id=gWxpdtQpiYV).*

Rectified flow emphasizes straight paths and can apply reflow to straighten the learned coupling further. “Straight” refers to trajectories under a coupling, not a guarantee of one-step high-quality generation with a finite model.

<details>
<summary><strong>PyTorch: train a conditional flow-matching vector field and integrate it</strong></summary>

```python
class ConditionalVelocity(nn.Module):
    def __init__(self, data_dim=64, hidden=128):
        super().__init__()
        self.label_embedding = nn.Embedding(10, 20)
        self.network = nn.Sequential(
            nn.Linear(data_dim + 1 + 20, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, data_dim),
        )

    def forward(self, points, times, labels):
        return self.network(torch.cat([points, times, self.label_embedding(labels)], dim=1))


seed_everything(1730)
velocity_model = ConditionalVelocity()
optimizer = torch.optim.AdamW(velocity_model.parameters(), lr=2e-3, weight_decay=1e-5)
generator = torch.Generator().manual_seed(1730)
for _ in range(1200):
    batch_indices = torch.randint(len(train_scaled), (192,), generator=generator)
    data, labels = train_scaled[batch_indices], train_y[batch_indices]
    source = torch.randn(data.shape, generator=generator)
    times = torch.rand((len(data), 1), generator=generator)
    path_points = (1 - times) * source + times * data
    target_velocity = data - source
    optimizer.zero_grad()
    flow_loss = F.mse_loss(velocity_model(path_points, times, labels), target_velocity)
    flow_loss.backward()
    optimizer.step()


@torch.no_grad()
def flow_sample(model, labels, steps, seed):
    generator = torch.Generator().manual_seed(seed)
    points = torch.randn((len(labels), 64), generator=generator)
    dt = 1.0 / steps
    for step in range(steps):
        times = torch.full((len(labels), 1), (step + 0.5) * dt)
        points = points + dt * model(points, times, labels)
    return points


flow_results = {}
for offset, steps in enumerate((1, 5, 20)):
    flow_samples = (flow_sample(velocity_model, evaluation_labels, steps, 1731) + 1) / 2
    flow_results[steps] = audit_samples(flow_samples, evaluation_labels)
    print({"Euler steps": steps,
           **{key: round(value, 3) if isinstance(value, float) else value
              for key, value in flow_results[steps].items()}})

assert math.isfinite(float(flow_loss))
```

</details>

The independent endpoint coupling creates crossing conditional paths; the regression learns their conditional average velocity. Better couplings, optimal-transport paths, reflow, and higher-order solvers can reduce curvature or integration error. The training MSE alone does not measure sample quality.


### **GANs, Diffusion, and Flow Matching Compared** {#gans-diffusion-flow-matching-compared}

| Property | GAN | Diffusion / score model | Flow matching |
|---|---|---|---|
| Learned object | generator through critic | denoiser, noise, score, or velocity target | time-dependent velocity field |
| Training signal | adversarial distribution comparison | supervised corruption target | supervised conditional path velocity |
| Likelihood | generally unavailable | bounds or ODE likelihood with extra machinery | CNF likelihood possible with divergence integration |
| Sampling | usually one generator pass | iterative reverse chain/SDE/ODE | iterative ODE solve |
| Conditioning | both players receive $c$ | conditional denoiser and guidance | conditional velocity field and guidance variants |
| Main strengths | fast sampling, sharp outputs | stable regression, coverage, flexible conditioning | simple simulation-free training, transport viewpoint |
| Main risks | mode collapse and game instability | many NFEs, schedule/solver errors | path/coupling choice and ODE discretization |

The chapter's numbers are not a leaderboard. Each model received a small but different optimization budget and architecture. The shared probe makes failure dimensions visible, yet its confidence and features favor what it learned from real digits. Fair comparison requires matched compute, tuning budget, sample count, repeated seeds, and a domain-valid evaluator.

Model choice depends on system requirements. A GAN remains attractive when one-pass latency dominates and adversarial training is manageable. Diffusion is attractive when coverage, conditioning, editing, and mature tooling matter more than iterative cost. Flow matching is attractive when direct velocity regression and ODE transport offer a useful quality-speed trade-off. Hybrid systems often combine latent autoencoders, adversarial reconstruction, diffusion or flow priors, and distillation.


### **Chapter Comparison and Summary** {#chapter-comparison-summary}

This chapter connected three modern generation paradigms through one dataset and one audit contract. The GAN sections exposed two-player optimization, non-saturating gradients, conditional interventions, WGAN-GP, and mode-coverage diagnostics. The diffusion sections derived direct noising, trained one multi-SNR conditional denoiser, converted epsilon/x0/v parameterizations, related discrete diffusion to SDE and ODE views, and tested CFG, latent compression, and reduced-step sampling. Flow matching then replaced reverse-noise prediction with velocity regression along chosen paths.

The central distinction is the learned field. GANs receive gradients from a moving critic. Diffusion models learn a score-equivalent denoising field over noise scales. Flow matching learns a transport velocity field. Sampling cost follows from that choice: one generator evaluation, many reverse denoising evaluations, or an ODE solver with a chosen number of function evaluations.

A practical review should verify the condition contract, data scaling, schedule or path, prediction parameterization, sampler equations, and model/solver compatibility. It should separate fidelity, coverage, conditional adherence, novelty, memorization, and latency; report seeds and compute; and avoid treating a classifier score or attractive sample grid as proof of distribution quality.

Chapter 17 moves from distribution generation to sequential decision making and world models. The generative components developed here become learned simulators, trajectory models, policies, or imagination mechanisms, but decision quality also depends on reward, uncertainty, exploration, and compounding model error.
